# Librería

In [1]:
# Manipulacion de datos
import pandas as pd
import numpy as np
import datetime
pd.set_option('display.max_columns', 200)
import json
import os
import unicodedata

import geopandas as gpd
from shapely.geometry import Point

from pathlib import Path

# Paths

In [2]:
csv_path = Path('/Users/jaydymarchan/Desktop/causalidad/data/01_raw/lluvias/')

# Funciones

In [3]:
pd.read_csv('/Users/jaydymarchan/Desktop/causalidad/data/01_raw/lluvias/201301010000Lluv.csv', encoding='latin-1').head(10)

,LON,LAT,EDO,CLAVE,ESTACION,ene-13
0,-102.309722,21.895000,AGS,AGSAG,"Aguascalientes, Ags.",58.50
1,-102.585833,22.177222,AGS,ALMAG,"Alamitos, Ags.",62.70
2,-102.464167,22.188611,AGS,ANVAG,"Cincuenta Aniversario, Ags.",54.40
3,-102.184167,21.738611,AGS,BRTAG,"San Bartolo, Ags.",59.00
4,-102.712222,21.849167,AGS,CALVILLO,"Calvillo, Ags. SMN*",32.82
5,-102.676944,21.997500,AGS,CDRAG,"La Codorniz, Ags.",45.00
6,-102.711667,21.836667,AGS,CLVAG,"Calvillo, Ags.",44.50
7,-101.992500,21.897778,AGS,CNSAG,"Los Conos, Ags.",54.81
8,-102.296944,22.362778,AGS,CSOAG,"Cosío, Ags.",49.01
9,-102.356944,22.121667,AGS,JCQAG,"Jocoque, Ags.",57.81


In [4]:
# .to_csv('/Users/jaydymarchan/Desktop/causalidad/data/02_processed/datos_sequia.csv', index=False, encoding='utf-8')

In [5]:
# Definir la ruta de la carpeta donde están las lluvias
carpeta_lluvias = Path('/Users/jaydymarchan/Desktop/causalidad/data/01_raw/lluvias/')
# Diccionario para obtener el número de entidad (cve_ent)
diccionario_edos = {
    'AGS': 1, 'BC': 2, 'BCN': 2, 'BCS': 3, 'CAMP': 4, 'COAH': 5, 
    'COL': 6, 'CHIS': 7, 'CHIH': 8, 'CMX': 9, 'DF': 9, 'CDMX': 9, 'DGO': 10, 
    'GTO': 11, 'GRO': 12, 'HGO': 13, 'JAL': 14, 'MEX': 15, 'MICH': 16, 
    'MOR': 17, 'NAY': 18, 'NL': 19, 'OAX': 20, 'PUE': 21, 'QRO': 22, 
    'QROO': 23, 'ROO': 23, 'SLP': 24, 'SIN': 25, 'SON': 26, 'TAB': 27, 
    'TAM': 28, 'TAMPS': 28, 'TLAX': 29, 'VER': 30, 'YUC': 31, 'ZAC': 32
}

# NUEVO: Diccionario para obtener el nombre completo a partir de la cve_ent
diccionario_nombres_edos = {
    1: 'Aguascalientes', 2: 'Baja California', 3: 'Baja California Sur', 
    4: 'Campeche', 5: 'Coahuila', 6: 'Colima', 7: 'Chiapas', 
    8: 'Chihuahua', 9: 'Ciudad de México', 10: 'Durango', 
    11: 'Guanajuato', 12: 'Guerrero', 13: 'Hidalgo', 14: 'Jalisco', 
    15: 'Estado de México', 16: 'Michoacán', 17: 'Morelos', 18: 'Nayarit', 
    19: 'Nuevo León', 20: 'Oaxaca', 21: 'Puebla', 22: 'Querétaro', 
    23: 'Quintana Roo', 24: 'San Luis Potosí', 25: 'Sinaloa', 
    26: 'Sonora', 27: 'Tabasco', 28: 'Tamaulipas', 29: 'Tlaxcala', 
    30: 'Veracruz', 31: 'Yucatán', 32: 'Zacatecas'
}

dataframes_lluvias = []

# Iterar sobre todos los archivos CSV en la carpeta
for archivo in carpeta_lluvias.glob('*.csv'):
    # Leer el archivo actual
    df_temp = pd.read_csv(archivo, encoding='latin-1')
    
    # 1. Identificar el nombre de la última columna
    nombre_col_lluvia = df_temp.columns[-1]
    
    # Renombrar esa columna a 'lluvia_mm'
    df_temp.rename(columns={nombre_col_lluvia: 'lluvia_mm'}, inplace=True)
    
    # 2. Limpiar la columna CLAVE
    df_temp['CLAVE'] = df_temp['CLAVE'].astype(str).str.strip()
    
    # 3. Extraer el nombre del municipio (lo que está antes de la primera coma)
    df_temp['municipio'] = df_temp['ESTACION'].str.split(',').str[0].str.strip()
    
    # 4. Convertir EDO a número usando el primer diccionario
    df_temp['EDO'] = df_temp['EDO'].astype(str).str.strip().str.upper()
    df_temp['cve_ent'] = df_temp['EDO'].map(diccionario_edos)
    
    # 5. NUEVO: Asignar el nombre completo de la entidad usando el segundo diccionario
    df_temp['nombre_entidad'] = df_temp['cve_ent'].map(diccionario_nombres_edos)
    
    # 6. Extracción de año y mes desde el nombre del archivo
    nombre_archivo = archivo.name
    df_temp['anio'] = pd.to_numeric(nombre_archivo[:4])
    df_temp['mes'] = pd.to_numeric(nombre_archivo[4:6])
    
    # Agregar el DataFrame procesado a la lista
    dataframes_lluvias.append(df_temp)

# Concatenar todos los DataFrames en uno solo maestro
df_lluvias_final = pd.concat(dataframes_lluvias, ignore_index=True)

# Mostrar el resultado final
display(df_lluvias_final.head())

,LON,LAT,EDO,CLAVE,ESTACION,lluvia_mm,municipio,cve_ent,nombre_entidad,anio,mes
0,-102.309722,21.895000,AGS,AGSAG,"Aguascalientes, Ags.",0.01,Aguascalientes,1.0,Aguascalientes,2018,4
1,-102.585833,22.177222,AGS,ALMAG,"Alamitos, Ags.",20.00,Alamitos,1.0,Aguascalientes,2018,4
2,-102.464167,22.188611,AGS,ANVAG,"Cincuenta Aniversario, Ags.",2.60,Cincuenta Aniversario,1.0,Aguascalientes,2018,4
3,-102.184167,21.738611,AGS,BRTAG,"San Bartolo, Ags.",10.00,San Bartolo,1.0,Aguascalientes,2018,4
4,-102.712222,21.849167,AGS,CALVILLO,"Calvillo, Ags. SMN*",5.30,Calvillo,1.0,Aguascalientes,2018,4


# Sanity check: cobertura de meses y años (2013–2025)

In [ ]:
# =========================================================
# SANITY CHECK: ¿están todos los meses y años 2013–2025?
# =========================================================
anio_ini, anio_fin = 2013, 2025

# --- 1) Cobertura segun los ARCHIVOS en la carpeta -------------------------
archivos = sorted(carpeta_lluvias.glob('*.csv'))
periodos_archivos = set()
for a in archivos:
    y, m = int(a.name[:4]), int(a.name[4:6])
    periodos_archivos.add((y, m))

# --- 2) Cobertura segun el DataFrame ya cargado ---------------------------
periodos_df = set(
    map(tuple, df_lluvias_final[['anio', 'mes']].drop_duplicates().to_numpy())
)

# --- 3) Rejilla esperada (13 años x 12 meses = 156 periodos) -------------
esperados = {(y, m) for y in range(anio_ini, anio_fin + 1) for m in range(1, 13)}

falt_archivos = sorted(esperados - periodos_archivos)
falt_df       = sorted(esperados - periodos_df)
extra_df      = sorted(periodos_df - esperados)          # periodos fuera de rango
solo_archivos = sorted(periodos_archivos - periodos_df)  # archivo existe pero no llegó al df

print(f"Periodos esperados      : {len(esperados)}  ({anio_ini}-01 a {anio_fin}-12)")
print(f"Periodos en archivos    : {len(periodos_archivos)}")
print(f"Periodos en df_lluvias  : {len(periodos_df)}")
print()
print(f"FALTAN en archivos ({len(falt_archivos)}):")
print("  " + (", ".join(f"{y}-{m:02d}" for y, m in falt_archivos) or "ninguno ✔"))
print()
print(f"FALTAN en df_lluvias_final ({len(falt_df)}):")
print("  " + (", ".join(f"{y}-{m:02d}" for y, m in falt_df) or "ninguno ✔"))
print()
print(f"Periodos FUERA de rango 2013–2025 en el df ({len(extra_df)}):")
print("  " + (", ".join(f"{y}-{m:02d}" for y, m in extra_df) or "ninguno ✔"))
print()
print(f"Archivos que NO aparecen en el df ({len(solo_archivos)}):")
print("  " + (", ".join(f"{y}-{m:02d}" for y, m in solo_archivos) or "ninguno ✔"))

# --- 4) Matriz visual año x mes (1 = presente en archivos, . = falta) ----
print("\nMatriz de cobertura (archivos)   1 = presente | . = falta")
print("año  " + " ".join(f"{m:02d}" for m in range(1, 13)))
for y in range(anio_ini, anio_fin + 1):
    fila = " ".join(" 1" if (y, m) in periodos_archivos else "  ." for m in range(1, 13))
    print(f"{y} {fila}")

# --- 5) Conteo de registros y estaciones por periodo ---------------------
resumen_periodos = (
    df_lluvias_final
    .groupby(['anio', 'mes'])
    .agg(n_registros=('lluvia_mm', 'size'),
         n_estaciones=('CLAVE', 'nunique'),
         lluvia_nula=('lluvia_mm', lambda s: s.isna().sum()))
    .reset_index()
    .sort_values(['anio', 'mes'])
)
display(resumen_periodos)

In [10]:
# 1. Cargar el shapefile geográfico
ruta_shp = '/Users/jaydymarchan/Desktop/causalidad/data/01_raw/datos_lat_longitud/2024_1_00_MUN.shp'


# 1. Cargar el shapefile geográfico
# ruta_shp = '/Users/jaydymarchan/Desktop/causalidad/data/01_raw/datos_lat_longitud/2024_1_00_ENT.shp'
gdf_poligonos = gpd.read_file(ruta_shp)

# --- CORRECCIÓN: Si el shapefile no tiene CRS asignado, se lo definimos (INEGI suele usar WGS84 por defecto) ---
if gdf_poligonos.crs is None:
    gdf_poligonos.set_crs("EPSG:4326", inplace=True)

# Asegurar que el sistema de coordenadas final sea el estándar (Lat/Lon WGS84)
if gdf_poligonos.crs != "EPSG:4326":
    gdf_poligonos = gdf_poligonos.to_crs("EPSG:4326")

# 2. Convertir tu tabla de lluvias en un GeoDataFrame
# (Asegúrate de que df_lluvias_final tenga las columnas 'LON' y 'LAT' previamente limpias)
geometria_estaciones = [Point(xy) for xy in zip(df_lluvias_final['LON'], df_lluvias_final['LAT'])]
gdf_estaciones = gpd.GeoDataFrame(df_lluvias_final, geometry=geometria_estaciones, crs="EPSG:4326")

# 3. Realizar el cruce espacial (Spatial Join)
gdf_cruce = gpd.sjoin(gdf_estaciones, gdf_poligonos, how="inner", predicate="within")

display(gdf_cruce.head())


,LON,LAT,EDO,CLAVE,ESTACION,lluvia_mm,municipio,cve_ent,nombre_entidad,anio,mes,geometry,index_right,CVEGEO,CVE_ENT,CVE_MUN,NOMGEO
0,-102.309722,21.895000,AGS,AGSAG,"Aguascalientes, Ags.",0.01,Aguascalientes,1.0,Aguascalientes,2018,4,POINT (-102.30972 21.895),2,01001,01,001,Aguascalientes
1,-102.585833,22.177222,AGS,ALMAG,"Alamitos, Ags.",20.00,Alamitos,1.0,Aguascalientes,2018,4,POINT (-102.58583 22.17722),1,01008,01,008,San José de Gracia
2,-102.464167,22.188611,AGS,ANVAG,"Cincuenta Aniversario, Ags.",2.60,Cincuenta Aniversario,1.0,Aguascalientes,2018,4,POINT (-102.46417 22.18861),1,01008,01,008,San José de Gracia
3,-102.184167,21.738611,AGS,BRTAG,"San Bartolo, Ags.",10.00,San Bartolo,1.0,Aguascalientes,2018,4,POINT (-102.18417 21.73861),2,01001,01,001,Aguascalientes
4,-102.712222,21.849167,AGS,CALVILLO,"Calvillo, Ags. SMN*",5.30,Calvillo,1.0,Aguascalientes,2018,4,POINT (-102.71222 21.84917),6,01003,01,003,Calvillo


In [29]:
df_final = gdf_cruce[['CVE_ENT', 'CVE_MUN',	'nombre_entidad', 'NOMGEO', 'anio',	'mes', 'lluvia_mm']]
df_lluvias_agrupadas = df_final.groupby(['CVE_ENT', 'CVE_MUN', 'nombre_entidad', 'NOMGEO','anio', 'mes']).agg({'lluvia_mm': 'sum'}).reset_index()
# ignorar la columna de cve_ent con el que viene los datos originales

In [31]:
# 1. Asegurar tipos de datos correctos
df_lluvias_agrupadas['anio'] = df_lluvias_agrupadas['anio'].astype(int)
df_lluvias_agrupadas['mes'] = df_lluvias_agrupadas['mes'].astype(int)

cols_base = ['CVE_ENT', 'CVE_MUN', 'nombre_entidad', 'NOMGEO']

# --- PASO EXTRA: Calcular la lluvia total de TODO el año anterior por municipio ---
# Sumamos los 12 meses de cada año
lluvia_anual_total = df_lluvias_agrupadas.groupby(
    cols_base + ['anio'], 
    as_index=False
)['lluvia_mm'].agg(lambda x: x.sum(min_count=1))

# Renombramos la columna para identificarla claramente
lluvia_anual_total.rename(columns={'lluvia_mm': 'lluvia_total_anual_mm'}, inplace=True)

# Creamos una copia desplazando el año +1 para que el año anterior quede listo para unirse por 'anio' actual
lluvia_anual_anterior = lluvia_anual_total.copy()
lluvia_anual_anterior['anio'] = lluvia_anual_anterior['anio'] + 1
lluvia_anual_anterior.rename(columns={'lluvia_total_anual_mm': 'lluvia_anio_anterior_mm'}, inplace=True)


# --- PROCESAMIENTO DE CICLOS (OI y PV) ---
resultados_ciclos = []
anos_unicos = range(2013, 2025)

for anio in anos_unicos:
    # Ciclo Otoño-Invierno (OI)
    condicion_oi = (
        ((df_lluvias_agrupadas['anio'] == anio - 1) & (df_lluvias_agrupadas['mes'].isin([11, 12]))) |
        ((df_lluvias_agrupadas['anio'] == anio) & (df_lluvias_agrupadas['mes'].isin([1, 2, 3, 4])))
    )
    df_oi = df_lluvias_agrupadas[condicion_oi]
    acumulado_oi = df_oi.groupby(cols_base, as_index=False)['lluvia_mm'].agg(lambda x: x.sum(min_count=1))
    acumulado_oi['anio'] = anio
    acumulado_oi['nomcicloproductivo'] = 'OI'
    acumulado_oi.rename(columns={'lluvia_mm': 'lluvia_acumulada_mm'}, inplace=True)
    
    # Ciclo Primavera-Verano (PV)
    condicion_pv = (
        (df_lluvias_agrupadas['anio'] == anio) & (df_lluvias_agrupadas['mes'].isin([4, 5, 6, 7, 8, 9]))
    )
    df_pv = df_lluvias_agrupadas[condicion_pv]
    acumulado_pv = df_pv.groupby(cols_base, as_index=False)['lluvia_mm'].agg(lambda x: x.sum(min_count=1))
    acumulado_pv['anio'] = anio
    acumulado_pv['nomcicloproductivo'] = 'PV'
    acumulado_pv.rename(columns={'lluvia_mm': 'lluvia_acumulada_mm'}, inplace=True)
    
    resultados_ciclos.append(acumulado_oi)
    resultados_ciclos.append(acumulado_pv)

df_lluvias_ciclos = pd.concat(resultados_ciclos, ignore_index=True)


# --- INTEGRAR LA LLUVIA DEL AÑO ANTERIOR ---
# Hacemos un merge utilizando las llaves geográficas y el año
df_lluvias_ciclos_con_lag = pd.merge(
    df_lluvias_ciclos,
    lluvia_anual_anterior,
    on=cols_base + ['anio'],
    how='left'
)

# Reordenar columnas para mayor legibilidad
df_lluvias_ciclos_con_lag = df_lluvias_ciclos_con_lag[
    cols_base + ['anio', 'nomcicloproductivo', 'lluvia_acumulada_mm', 'lluvia_anio_anterior_mm']
]

display(df_lluvias_ciclos_con_lag.head(10))

,CVE_ENT,CVE_MUN,nombre_entidad,NOMGEO,anio,nomcicloproductivo,lluvia_acumulada_mm,lluvia_anio_anterior_mm
0,01,001,Aguascalientes,Aguascalientes,2013,OI,238.34,NaN
1,01,002,Aguascalientes,Asientos,2013,OI,59.90,NaN
2,01,003,Aguascalientes,Calvillo,2013,OI,165.07,NaN
3,01,004,Aguascalientes,Cosío,2013,OI,49.03,NaN
4,01,006,Aguascalientes,Pabellón de Arteaga,2013,OI,118.62,NaN
5,01,007,Aguascalientes,Rincón de Romos,2013,OI,52.30,NaN
6,01,008,Aguascalientes,San José de Gracia,2013,OI,172.10,NaN
7,01,009,Aguascalientes,Tepezalá,2013,OI,51.02,NaN
8,01,010,Aguascalientes,El Llano,2013,OI,293.91,NaN
9,02,001,Baja California,Ensenada,2013,OI,294.57,NaN


In [32]:
df_lluvias_ciclos_con_lag.to_csv('/Users/jaydymarchan/Desktop/causalidad/data/02_processed/datos_lluvia.csv', index=False, encoding='utf-8')